# Homework 1: Retrieval-Based Chatbot (Russian)
Minimal, clean notebook based on seminar code and adapted for Google Colab.


In [ ]:
# For Google Colab. Skip if packages are already installed locally.
!pip install -q transformers datasets sentence-transformers rank-bm25 tabulate
!pip install -q faiss-gpu 2>/dev/null || pip install -q faiss-cpu


In [ ]:
import time
import re
import urllib.request
import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tabulate import tabulate

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU")


## 1) Data
Download `Master and Margarita` text and build a retrieval corpus.


In [ ]:
url = "http://lib.ru/BULGAKOW/mm7.txt"
urllib.request.urlretrieve(url, "master_margarita.txt")

try:
    with open("master_margarita.txt", "r", encoding="cp1251") as f:
        raw_text = f.read()
except UnicodeDecodeError:
    with open("master_margarita.txt", "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

# Basic cleanup
raw_text = raw_text.replace("\r", "\n")
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text)

# Split into paragraphs and keep meaningful ones
paragraphs = [p.strip() for p in raw_text.split("\n\n")]
paragraphs = [p for p in paragraphs if len(p) >= 120 and not p.startswith("http")]

N_DOCS = min(10000, len(paragraphs))
documents = paragraphs[:N_DOCS]
titles = [f"Passage {i}" for i in range(N_DOCS)]

# Query set for automatic evaluation (self-retrieval setup)
N_QUERIES = min(200, N_DOCS)
queries = [re.split(r"[.!?]\s+", documents[i])[0][:120] for i in range(N_QUERIES)]
ground_truth = list(range(N_QUERIES))

print(f"Documents: {len(documents)}")
print(f"Queries: {len(queries)}")
if queries:
    print(f"Example query: {queries[0]}")


## 2) Keyword Baseline (BM25)

In [ ]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)


def bm25_search(query: str, top_k: int = 10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices.tolist(), scores[top_indices].tolist()


def bm25_answer_user(query: str, top_k: int = 3):
    idxs, scores = bm25_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 3) Dense Retrieval (Sentence-Transformers)

In [ ]:
retriever = SentenceTransformer("intfloat/multilingual-e5-small", device=str(device))

docs_with_prefix = [f"passage: {doc}" for doc in documents]

t0 = time.time()
doc_embeddings = retriever.encode(
    docs_with_prefix,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {doc_embeddings.shape}")
print(f"Encoding time: {time.time() - t0:.2f} sec")

In [ ]:
def dense_search(query: str, top_k: int = 10):
    query_emb = retriever.encode(f"query: {query}", normalize_embeddings=True)
    scores = doc_embeddings @ query_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices.tolist(), scores[top_indices].tolist()


def dense_answer_user(query: str, top_k: int = 3):
    idxs, scores = dense_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 4) FAISS Index

In [ ]:
d = doc_embeddings.shape[1]
index_flat = faiss.IndexFlatIP(d)
index_flat.add(doc_embeddings.astype(np.float32))

print(f"FAISS index vectors: {index_flat.ntotal}")

In [ ]:
def faiss_search(query: str, top_k: int = 10):
    query_emb = retriever.encode([f"query: {query}"], normalize_embeddings=True).astype(np.float32)
    scores, indices = index_flat.search(query_emb, top_k)
    return indices[0].tolist(), scores[0].tolist()


def faiss_answer_user(query: str, top_k: int = 3):
    idxs, scores = faiss_search(query, top_k=top_k)
    return [
        {
            "rank": i + 1,
            "score": float(scores[i]),
            "title": titles[idx],
            "text": documents[idx],
        }
        for i, idx in enumerate(idxs)
    ]

## 5) Metrics

In [ ]:
def recall_at_k(results, ground_truth, k: int) -> float:
    hits = sum(1 for i, res in enumerate(results) if ground_truth[i] in res[:k])
    return hits / len(results)


def mrr(results, ground_truth) -> float:
    rr_sum = 0.0
    for i, res in enumerate(results):
        try:
            rank = res.index(ground_truth[i]) + 1
            rr_sum += 1.0 / rank
        except ValueError:
            pass
    return rr_sum / len(results)


def ndcg_at_k(results, ground_truth, k: int) -> float:
    ndcg_sum = 0.0
    for i, res in enumerate(results):
        dcg = 0.0
        for rank, doc_idx in enumerate(res[:k], 1):
            if doc_idx == ground_truth[i]:
                dcg += 1.0 / np.log2(rank + 1)
        ndcg_sum += dcg
    return ndcg_sum / len(results)

## 6) Evaluate BM25 vs Dense vs FAISS

In [ ]:
bm25_results = []
dense_results = []
faiss_results = []

for q in queries:
    bm25_idxs, _ = bm25_search(q, top_k=20)
    dense_idxs, _ = dense_search(q, top_k=20)
    faiss_idxs, _ = faiss_search(q, top_k=20)
    bm25_results.append(bm25_idxs)
    dense_results.append(dense_idxs)
    faiss_results.append(faiss_idxs)

rows = []
all_methods = {
    "BM25": bm25_results,
    "Dense (brute-force)": dense_results,
    "Dense + FAISS Flat": faiss_results,
}

for name, res in all_methods.items():
    rows.append([
        name,
        f"{mrr(res, ground_truth):.3f}",
        f"{ndcg_at_k(res, ground_truth, 10):.3f}",
        f"{recall_at_k(res, ground_truth, 1):.3f}",
        f"{recall_at_k(res, ground_truth, 5):.3f}",
        f"{recall_at_k(res, ground_truth, 10):.3f}",
    ])

print(tabulate(rows, headers=["Method", "MRR", "NDCG@10", "R@1", "R@5", "R@10"], tablefmt="rounded_outline"))

## 7) Inference Demo (with similarity scores)


In [ ]:
user_query = "Понтий Пилат"

print("BM25:")
for row in bm25_answer_user(user_query, top_k=3):
    print(f"{row['rank']}. score={row['score']:.3f} | {row['title']}")

print("\nDense + FAISS:")
for row in faiss_answer_user(user_query, top_k=3):
    print(f"{row['rank']}. score={row['score']:.3f} | {row['title']}")


## 8) Qualitative Comparison: Vector Search vs Keyword Search
Use your own 3-5 queries and write short conclusions in markdown.


In [ ]:
test_queries = [
    "Понтий Пилат и прокуратор",
    "бал у сатаны",
    "Мастер и Маргарита",
]

for q in test_queries:
    print("=" * 90)
    print(f"Query: {q}")

    bm25_top1 = bm25_answer_user(q, top_k=1)[0]
    faiss_top1 = faiss_answer_user(q, top_k=1)[0]

    print(f"BM25 top-1:  score={bm25_top1['score']:.3f} | {bm25_top1['title']}")
    print(f"Dense top-1: score={faiss_top1['score']:.3f} | {faiss_top1['title']}")
